In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from pathlib import Path
 
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)

In [3]:
DATA_DIR    = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")               
MODEL_NAME  = "microsoft/deberta-v3-base"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN     = 256
BATCH_SIZE  = 4
EPOCHS      = 3
LR          = 1e-5
WARMUP_FRAC = 0.1
OPTION_COLS = ["A", "B", "C", "D", "E"]
 
print(f"Device: {DEVICE}")
print(f"Model : {MODEL_NAME}")

Device: cuda
Model : microsoft/deberta-v3-base


In [4]:
def apk(actual, predicted, k=3):
    if not actual:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)
 
def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
 
print(f"Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

Train: 2000 rows  |  Test: 500 rows


In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.has_labels = has_labels
        self.label_map  = {c: i for i, c in enumerate(OPTION_COLS)}
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        question = str(row["prompt"])
        choices  = [str(row[c]) for c in OPTION_COLS]
 
        # Tokenise all 5 (question, choice) pairs at once
        enc = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        # enc values shape: (5, max_len)
        item = {k: v for k, v in enc.items()}
 
        if self.has_labels:
            item["labels"] = torch.tensor(
                self.label_map[str(row["answer"])], dtype=torch.long
            )
        return item

In [7]:
print("\nLoading tokenizer and model weights from HuggingFace...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
 
train_dataset = MCQDataset(train_df, tokenizer, has_labels=True)
test_dataset  = MCQDataset(test_df,  tokenizer, has_labels=False)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


Loading tokenizer and model weights from HuggingFace...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                

In [8]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)

In [9]:
print("\n" + "="*50)
print("Fine-tuning DeBERTa-v3-base …")
print("="*50)
 
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
 
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)        # (B, 5, L)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
 
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss   = outputs.loss
        logits = outputs.logits                               # (B, 5)
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
 
        total_loss += loss.item()
        preds       = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
 
        if (step + 1) % 50 == 0:
            print(f"  step {step+1}/{len(train_loader)}  "
                  f"loss={total_loss/(step+1):.4f}  "
                  f"acc={correct/total:.4f}")
 
    epoch_loss = total_loss / len(train_loader)
    epoch_acc  = correct / total
    print(f"\nEpoch {epoch+1}/{EPOCHS} complete — "
          f"avg loss: {epoch_loss:.4f}  train acc: {epoch_acc:.4f}\n")


Fine-tuning DeBERTa-v3-base …


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  step 50/500  loss=nan  acc=0.1550
  step 100/500  loss=nan  acc=0.1600
  step 150/500  loss=nan  acc=0.1633
  step 200/500  loss=nan  acc=0.1737
  step 250/500  loss=nan  acc=0.1790
  step 300/500  loss=nan  acc=0.1775
  step 350/500  loss=nan  acc=0.1793
  step 400/500  loss=nan  acc=0.1856
  step 450/500  loss=nan  acc=0.1906
  step 500/500  loss=nan  acc=0.1850

Epoch 1/3 complete — avg loss: nan  train acc: 0.1850

  step 50/500  loss=nan  acc=0.1750
  step 100/500  loss=nan  acc=0.1850
  step 150/500  loss=nan  acc=0.1800
  step 200/500  loss=nan  acc=0.1963
  step 250/500  loss=nan  acc=0.1870
  step 300/500  loss=nan  acc=0.1858
  step 350/500  loss=nan  acc=0.1879
  step 400/500  loss=nan  acc=0.1862
  step 450/500  loss=nan  acc=0.1856
  step 500/500  loss=nan  acc=0.1845

Epoch 2/3 complete — avg loss: nan  train acc: 0.1845

  step 50/500  loss=nan  acc=0.1800
  step 100/500  loss=nan  acc=0.1950
  step 150/500  loss=nan  acc=0.1833
  step 200/500  loss=nan  acc=0.1850
  s

In [10]:
def predict_top3(loader):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.cpu().numpy())  # (B, 5)
 
    logits = np.vstack(all_logits)   # (N, 5)
    preds  = []
    for row in logits:
        ranked = [OPTION_COLS[i] for i in np.argsort(row)[::-1]]
        preds.append(ranked[:3])
    return logits, preds

In [11]:
print("Evaluating on training set …")
_, train_preds = predict_top3(DataLoader(train_dataset, batch_size=BATCH_SIZE))
train_map3     = mapk(train_df["answer"].tolist(), train_preds)
print(f"Train MAP@3: {train_map3:.4f}")

Evaluating on training set …
Train MAP@3: 0.3280


In [12]:
print("\nGenerating test predictions …")
_, test_preds = predict_top3(test_loader)
 
submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
submission.to_csv("submission_deberta.csv", index=False)
 
print("\n✅  submission_deberta.csv saved!")
print(submission.head(10).to_string(index=False))


Generating test predictions …

✅  submission_deberta.csv saved!
 ID Prediction
  1      E D C
  2      E D C
  3      E D C
  4      E D C
  5      E D C
  6      E D C
  7      E D C
  8      E D C
  9      E D C
 10      E D C


In [13]:
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample_submission.to_csv("submission.csv", index = False)